# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [4]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [5]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [6]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/28/connecting-my-cou

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [7]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [8]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [9]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/11/11/ai-live-event/
https://edwarddonner.com/2025/11/11/ai-live-event/
https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/
htt

In [10]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [11]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [12]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [13]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 7 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'},
  {'type': 'Patent',
   'url': 'https://patents.google.com/patent/US20210049536A1/'}]}

In [14]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Zhihu page', 'url': 'https://www.zhihu.com/org/huggingface'},
  {'type': 'Community forum', 'url': 'https://discuss.huggingface.co'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [15]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [16]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 1M+ models
Trending on
this week
Models
Tongyi-MAI/Z-Image-Turbo
Updated
7 days ago
•
286k
•
2.73k
zai-org/GLM-4.6V-Flash
Updated
6 days ago
•
84.2k
•
435
microsoft/VibeVoice-Realtime-0.5B
Updated
3 days ago
•
143k
•
848
mistralai/Devstral-Small-2-24B-Instruct-2512
Updated
about 21 hours ago
•
21.4k
•
348
zai-org/GLM-4.6V
Updated
6 days ago
•
3.68k
•
306
Browse 1M+ models
Spaces
Running
on
Zero
580
Z Image Turbo
🖼
580
Generate stunning images from text prompts
Running
on
Zero
MCP
Featured
1.38k
Z Image Turbo
🏃
1.38k
Generate images from text prompts
Running
on
Zero
MCP
144

In [29]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [28]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [19]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 4 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 1M+ models\nTrending on\nthis week\nModels\nTongyi-MAI/Z-Image-Turbo\nUpdated\n7 days ago\n•\n286k\n•\n2.73k\nzai-org/GLM-4.6V-Flash\nUpdated\n6 days ago\n•\n84.2k\n•\n435\nmicrosoft/VibeVoice-Realtime-0.5B\nUpdated\n3 days ago\n•\n143k\n•\n848\nmistralai/Devstral-Small-2-24B-Instruct-2512\nUpdated\nabout 22 hours ago\n•\n21.4k\n•\n348\nzai-org/GLM-4.6V\nUpdated\n6 days ago\n•\n3.68k\n•\n306\nBrowse 1M+ models\nSpaces\nRunning\non\nZ

In [20]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [21]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 4 relevant links


# Hugging Face Brochure

---

## Who We Are  
**Hugging Face** is the AI community building the future. Founded in 2016 and headquartered in Paris, France, we are a rapidly growing, privately held software development company with a passionate team of 51-200 employees. Our mission is to democratize and open the future of artificial intelligence, focusing primarily on machine learning, natural language processing (NLP), and deep learning.

---

## What We Do  
We offer a vibrant collaboration platform—the **Hugging Face Hub**—where machine learning engineers, data scientists, and AI enthusiasts can create, discover, and share over 1 million open-source models, datasets, and applications across various AI modalities including text, images, audio, video, and even 3D.

Our platform empowers users to:  
- Host unlimited public models, datasets, and applications.  
- Collaborate effectively with the global AI community.  
- Explore cutting-edge machine learning models including transformers and reinforcement learning applications.  
- Build and showcase their ML portfolios publicly to foster recognition and career growth.

Alongside open community offerings, we provide robust **paid Compute and Enterprise solutions** designed to accelerate ML workflows for teams and organizations.

---

## Our Community & Customers  
Hugging Face thrives as a dynamic ecosystem with a fast-growing community of ML practitioners, researchers, and developers. We support both individual users and enterprise clients from startups to large organizations looking to leverage AI responsibly and transparently.

Our users benefit from:  
- Access to an extensive library with 1M+ models and 250k+ datasets.  
- Interactive apps and Spaces that demonstrate live AI capabilities.  
- Continuous updates and innovations shared openly within the community.

---

## Company Culture  
At Hugging Face, we believe in:  
- **Open and Ethical AI:** We strive for transparency and ethical standards in AI development.  
- **Collaboration:** The power of a supportive, fun, and inclusive community driving shared progress.  
- **Innovation & Learning:** Encouraging experimentation, iteration, and continuous learning at the frontier of AI research.  
- **Impact:** Enabling users worldwide to build the AI tools of tomorrow.

Our community-driven spirit is reflected in our lively forums, active Discord channels, and continuous engagement both inside and outside the organization.

---

## Careers & Opportunities  
Hugging Face is on the lookout for talent who are passionate about AI and eager to contribute to open-source projects that shape the industry. We offer a creative environment that balances fast-paced innovation with a collaborative and supportive culture.  

Current job openings encompass roles across:  
- Machine Learning Engineering  
- Software Development  
- Research Science  
- Community Management  
- Product Development  

If you are excited to work with cutting-edge AI tech in a mission-driven company, explore our latest openings on [Hugging Face Careers](https://huggingface.co/careers).

---

## Quick Facts  
- Founded: 2016  
- Employees: 51-200  
- Headquarters: Paris, France  
- Industry: Software Development, AI, Machine Learning  
- Specialty: Natural Language Processing, Transformers, Open-source ML tools

---

## How to Connect  
Website: [https://huggingface.co](https://huggingface.co)  
Community: Active on GitHub, Twitter, LinkedIn, Discord  
Contact: Join the vibrant AI community and start collaborating today!

---

**Hugging Face** – Building the future of AI, together.  
Empowering the next generation to learn, share, and innovate in machine learning.

---

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [25]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [26]:
stream_brochure("YCombinator", "https://www.ycombinator.com/")

Selecting relevant links for https://www.ycombinator.com/ by calling gpt-5-nano
Found 15 relevant links


# Y Combinator Brochure

---

## Who We Are

**Y Combinator (YC)** is the world's leading startup accelerator, dedicated to helping founders create products that people want. With over **5,000 funded startups** and a combined valuation exceeding **$800 billion**, YC has established itself as the launchpad for many of the world's most successful startups.

---

## What We Do

At YC, we believe in empowering founders at the earliest stages of their entrepreneurial journey. We provide funding, mentorship, and an intense, immersive 3-month program designed to elevate startups, whether they are at the idea phase or already launched.

Our startups gain:
- A better product with more users
- Increased fundraising options and advantages
- Access to a world-class network of investors and experienced founders

---

## Our Program

YC runs **four batches a year** (winter, spring, summer, fall), each lasting 3 months. During this period, founders engage deeply with YC’s community and resources to build and scale their companies rapidly. Founders often describe the experience as the most productive and transformational period of their careers.

Key highlights:
- **Dedicated YC Partners:** Each startup is assigned a partner who has mentored hundreds of companies, providing invaluable advice via in-person meetings, email, and Slack.
- **Intense, Collaborative Atmosphere:** The entire community—other founders, alumni, investors, and YC staff—works together to help startups thrive.
- **Demo Day:** An exclusive event where founders present their startups to a curated group of investors.

---

## Our People

YC is led by a team of experienced startup founders and professionals who have themselves built successful companies. 

- **Garry Tan, President & CEO** – Former YC partner who created key parts of the YC founder experience, including Bookface and Demo Day platforms, now leads YC with a focus on expanding its reach and impact.

Our partners and team members bring unmatched startup experience and are committed to helping founders succeed.

---

## Our Community & Network

- Access to **over 9,000 YC alumni** through *Bookface,* YC’s private social network exclusively for founders — providing a rich resource for advice, connections, and collaboration.
- An **investor network** with $85 billion raised by YC companies to date from top-tier investors worldwide.
- **Founder directory and startup directory** to help startups find partners, advisors, customers, or talent.

---

## Careers at Y Combinator

YC offers job opportunities across multiple functions:
- Engineering
- Operations
- Marketing
- Sales
- Internships

For those passionate about startups and innovation, working at YC means being at the heart of the startup ecosystem, supporting the next generation of revolutionary companies.

---

## Why Join or Invest in YC?

- Proven success with companies that often become billion-dollar businesses
- Access to cutting-edge startups and entrepreneurs
- A founder-first culture focused on long-term partnership and support
- Invaluable resources for fundraising and growth

---

## Apply and Get Involved

Whether you are a founder, investor, or startup enthusiast, YC welcomes you to become part of its thriving community. Apply for our upcoming batch, find a co-founder, or explore internships via the YC website.

---

**Y Combinator**  
*Make something people want.*

Website: [ycombinator.com](https://www.ycombinator.com)  
Apply for the next batch: [Apply to YC](https://www.ycombinator.com/apply)

---

*Join Y Combinator and be at the forefront of the future of innovation.*

In [30]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("YCombinator", "https://www.ycombinator.com/")

Selecting relevant links for https://www.ycombinator.com/ by calling gpt-5-nano
Found 21 relevant links


# Welcome to Y Combinator: Where Startups Go From Zero to Hero 🚀

## Who Are We?  
Y Combinator (YC for short) is the legendary rocketship launching pad for ambitious startups. We don’t just *fund* companies — we help founders **make something people want** and turn that spark into a roaring wildfire. Since Day One, we've funded over **5,000 startups** with a mind-blowing combined valuation of **$800 billion**. Yes, billion with a B.  

## What’s the YC Magic Formula?  
- **We help founders at their earliest stages** — no matter your age, background, or whether your office is a garage or a coffee shop.
- **Dedicated YC Partners** assigned to you. Each partner is a seasoned veteran who has mentored hundreds of successful companies, armed with the ultimate startup playbook.
- **Access to an insane investor network.** Our startups have raised a staggering $85 billion from the crème de la crème of startup investors worldwide.
- **Private founder social network — Bookface.** Imagine a Hogwarts of innovation, where over 9,000 YC alumni conveniently share wisdom, advice, and job leads.
- **Launch Day & Demo Day:** Where your dreams get a stage and the spotlight shines bright, ready to attract investors, press, and future customers.

## Our Culture: Nerdy, Bold & Supportive  
YC is run by former startup founders who’ve been in the trenches (and lived to tell the tale). We know the sleepless nights, the caffeine binges, the endless pivoting — and we’re here to make that journey less lonely, more focused, and wildly successful.

Expect direct mentorship, open email/slack lines, and a community that’s as hungry for success as you are. Whether you want to build a billion-dollar tech unicorn or solve quirky niche problems, YC’s culture screams: **“Make something people truly want. We’ll help you get there.”**

## For Founders & Future Success Stories  
- You’re an early-stage innovator with a crazy idea?  
- You want an unfair advantage in fundraising?  
- You crave mentorship from veterans who’ve “been there, scaled that”?  
- You want to plug into the most powerful startup network on the planet?  

**Apply for the X2026 batch now!** And remember, the application is just the start of your transformation.

## Careers at YC: Join the Startup Whisperers  
Passionate about startups? YC is always on the lookout for rockstars in:  
- Engineering  
- Operations  
- Marketing  
- Sales  

Or maybe you want to rock an internship that will make your resume the envy of every startup junkie. Join us, and help build the future of startups *from the inside*.

## Investors, Rejoice  
Looking to back the next big thing? YC’s founder database plus the investor network means you get insider access to the hottest innovations before they become buzzwords.

## In Summary:  
Y Combinator is the place where startup dreams get upgraded to market realities — with expert guidance, legendary networks, and community wisdom to spare. Whether you're a founder, investor, or aspiring YC team member, here’s your VIP pass to the startup universe.  

---

**Y Combinator**  
*Make something people want.*  
Apply today, build tomorrow, disrupt forever.  

**P.S.** We promise not to make you drink any fermented kale smoothie (unless you want to). 😉

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>